# 2 — Apprentissage supervisé

Cible : t2m décalée d'un mois, discrétisée en **3 intervalles de même largeur**. Split **chronologique** train / test.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
from src import supervised as sup
from src import config as C

clean = pd.read_parquet('../data/processed/era5_clean.parquet')
frame = sup.build_frame(clean)
frame[['time', 't2m', 't2m_lag1', 't2m_target']].head()

## Split chronologique puis discrétisation

Les bornes sont calculées **sur le train seul** : les découper sur l'ensemble complet ferait fuir l'information du futur.

In [ ]:
train, test, edges = sup.prepare(clean)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
train['classe'].value_counts().reindex(C.BIN_LABELS).plot.bar(ax=ax, title='Répartition des classes (train)');

## Modèles

In [ ]:
models = sup.evaluate(train, test)

In [ ]:
import numpy as np

imp = pd.Series(models['foret'].feature_importances_, index=sup.FEATURES).sort_values()
imp.plot.barh(figsize=(7, 4), title='Importance des variables (forêt aléatoire)');

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(
    models['foret'], test[sup.FEATURES], test['classe'],
    labels=C.BIN_LABELS, normalize='true', cmap='Blues'
);